# Прогноз клиентского потока и потребности касс

In [ ]:
!pip install statsmodels

In [ ]:
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from prefect.blocks.system import Secret
from sqlalchemy import text
from statsmodels.tsa.statespace.sarimax import SARIMAX
from toolbox import oracle

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. Параметры запуска

In [ ]:
SOURCE_TABLE = "AIDA2.AIDA_TRS_DTM_CASHOP@aida"
HISTORY_MONTHS = 18
ACTIVE_LOOKBACK_MONTHS = 1
DATA_LAG_DAYS = 2

RUN_DATE = pd.Timestamp.today().normalize()
MAX_DATA_DATE = RUN_DATE - pd.Timedelta(days=DATA_LAG_DAYS)
DATE_FROM = (MAX_DATA_DATE - pd.DateOffset(months=HISTORY_MONTHS)).date().isoformat()
DATE_TO_EXCLUSIVE = (MAX_DATA_DATE + pd.Timedelta(days=1)).date().isoformat()
EXCLUDE_DATE_RANGES = [("2025-09-01", "2025-11-01")]

FORECAST_DAYS = 30
INITIAL_TRAIN_DAYS = 60
BACKTEST_STEP_DAYS = 7
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_QUANTILE = 0.15
SARIMA_NO_ERROR_HISTORY_ADJUSTMENT = 0.15
MIN_SARIMA_DAYS = 90
MIN_BACKTEST_ERROR_BLOCKS = 5

SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 7)

MIN_OBSERVATIONS = MIN_SARIMA_DAYS
CASH_NEED_CLIP_LOWER = None
CASH_NEED_CLIP_UPPER = 0.0
CASHDESK_FILTER = None

## 2. Загрузка данных из Oracle

In [ ]:
USERNAME_CDW = "sb_analytics"

PASSWORD_CDW = (await Secret.load("pass-sb-analytics")).get()
engine_cdw = oracle.create_engine_cdw(USERNAME_CDW, PASSWORD_CDW)

In [ ]:
query = f"""
select
    atdtmco_cashdesk_name,
    atdtmco_calday,
    atdtmco_saldo_turn,
    atdtmco_ns
from {SOURCE_TABLE}
where atdtmco_calday >= date '{DATE_FROM}'
  and atdtmco_calday < date '{DATE_TO_EXCLUSIVE}'
"""

for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
    query += (
        f"\n  and not ("
        f"atdtmco_calday >= date '{exclude_start}' "
        f"and atdtmco_calday < date '{exclude_end}'"
        f")"
    )

if CASHDESK_FILTER:
    escaped_names = ", ".join(["'" + name.replace("'", "''") + "'" for name in CASHDESK_FILTER])
    query += f"\n  and atdtmco_cashdesk_name in ({escaped_names})"

with engine_cdw.connect() as conn:
    raw_df = pd.read_sql(text(query), conn)

raw_df.columns = raw_df.columns.str.lower()
raw_df["atdtmco_calday"] = pd.to_datetime(raw_df["atdtmco_calday"])

print(raw_df.shape)
raw_df.head()

## 3. Подготовка дневных рядов

In [ ]:
daily_df = (
    raw_df
    .assign(calday=lambda df: df["atdtmco_calday"].dt.normalize())
    .groupby(["atdtmco_cashdesk_name", "calday"], as_index=False)
    .agg(
        saldo_turn=("atdtmco_saldo_turn", "sum"),
        ns_daily_min=("atdtmco_ns", "min"),
    )
)

daily_df["cash_need"] = daily_df["ns_daily_min"].clip(upper=0)
daily_df = daily_df.sort_values(["atdtmco_cashdesk_name", "calday"]).reset_index(drop=True)

active_date_from = MAX_DATA_DATE - pd.DateOffset(months=ACTIVE_LOOKBACK_MONTHS)
active_cashdesks = daily_df.loc[
    (daily_df["calday"] >= active_date_from)
    & (daily_df["calday"] <= MAX_DATA_DATE),
    "atdtmco_cashdesk_name",
].drop_duplicates()

inactive_cashdesks_df = (
    daily_df.loc[~daily_df["atdtmco_cashdesk_name"].isin(active_cashdesks), ["atdtmco_cashdesk_name"]]
    .drop_duplicates()
    .assign(score_date=RUN_DATE, report_date=MAX_DATA_DATE)
)
daily_df = daily_df[daily_df["atdtmco_cashdesk_name"].isin(active_cashdesks)].reset_index(drop=True)

print(daily_df.shape)
daily_df.head()

## 4. Вспомогательные функции

In [ ]:
@dataclass
class SarimaConfig:
    order: tuple[int, int, int]
    seasonal_order: tuple[int, int, int, int]
    maxiter: int = 200


SARIMA_CONFIG = SarimaConfig(
    order=SARIMA_ORDER,
    seasonal_order=SARIMA_SEASONAL_ORDER,
)


def make_regular_daily_series(
    cashdesk_df: pd.DataFrame,
    value_col: str,
    fill_method: str,
) -> pd.Series:
    series = (
        cashdesk_df
        .set_index("calday")[value_col]
        .sort_index()
        .asfreq("D")
        .astype(float)
    )

    if fill_method == "zero":
        return series.fillna(0.0)
    if fill_method == "ffill":
        return series.ffill().bfill()
    if fill_method == "interpolate":
        return series.interpolate(method="time").ffill().bfill()

    raise ValueError(f"Неизвестный fill_method: {fill_method}")


def fit_sarima(y: pd.Series, config: SarimaConfig):
    model = SARIMAX(
        y,
        order=config.order,
        seasonal_order=config.seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit(disp=False, maxiter=config.maxiter)


def sarima_point_forecast(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
) -> pd.DataFrame:
    result = fit_sarima(y, config)
    forecast_result = result.get_forecast(steps=steps)
    forecast_mean = forecast_result.predicted_mean

    output = pd.DataFrame(
        {"forecast_mean": forecast_mean.values},
        index=forecast_mean.index,
    )
    output.index.name = "forecast_date"
    return output.reset_index()


def make_future_index(y: pd.Series, steps: int) -> pd.DatetimeIndex:
    return pd.date_range(y.index.max() + pd.Timedelta(days=1), periods=steps, freq="D")


def weekday_point_forecast(y: pd.Series, steps: int) -> pd.DataFrame:
    future_index = make_future_index(y, steps)
    global_value = float(y.dropna().median())
    weekday_values = y.dropna().groupby(y.dropna().index.dayofweek).median()
    values = [weekday_values.get(day.dayofweek, global_value) for day in future_index]
    output = pd.DataFrame(
        {"forecast_mean": values},
        index=future_index,
    )
    output.index.name = "forecast_date"
    return output.reset_index()


def weekday_quantile_forecast(y: pd.Series, steps: int, quantile: float) -> pd.DataFrame:
    future_index = make_future_index(y, steps)
    y_clean = y.dropna()
    global_quantile = float(np.quantile(y_clean.values, quantile))
    global_mean = float(y_clean.mean())
    weekday_quantiles = y_clean.groupby(y_clean.index.dayofweek).quantile(quantile)
    weekday_means = y_clean.groupby(y_clean.index.dayofweek).mean()
    quantile_values = [weekday_quantiles.get(day.dayofweek, global_quantile) for day in future_index]
    mean_values = [weekday_means.get(day.dayofweek, global_mean) for day in future_index]
    if CASH_NEED_CLIP_UPPER is not None:
        quantile_values = np.minimum(quantile_values, CASH_NEED_CLIP_UPPER)
        mean_values = np.minimum(mean_values, CASH_NEED_CLIP_UPPER)
    output = pd.DataFrame(
        {
            "forecast_mean": mean_values,
            "forecast_quantile": quantile_values,
        },
        index=future_index,
    )
    output.index.name = "forecast_date"
    return output.reset_index(), np.empty((0, steps)), np.empty((0, steps))


def collect_backtest_error_blocks(
    y: pd.Series,
    steps: int,
    initial_train_days: int,
    step_days: int,
    config: SarimaConfig,
) -> np.ndarray:
    error_blocks = []
    max_train_end = len(y) - steps

    for train_end in range(initial_train_days, max_train_end + 1, step_days):
        train = y.iloc[:train_end]
        test = y.iloc[train_end:train_end + steps]

        if train.notna().sum() < initial_train_days or len(test) < steps:
            continue

        try:
            fitted_model = fit_sarima(train, config)
            forecast = fitted_model.get_forecast(steps=steps).predicted_mean
            error = test.values - forecast.values

            if np.isfinite(error).all():
                error_blocks.append(error)
        except Exception as exc:
            print(f"Backtest iteration failed at train_end={train_end}: {exc}")
            continue

    if not error_blocks:
        raise ValueError("Не удалось собрать ни одного блока ошибок backtest.")

    return np.vstack(error_blocks)


def forecast_with_error_bootstrap(
    y: pd.Series,
    steps: int,
    initial_train_days: int,
    step_days: int,
    bootstrap_iterations: int,
    quantile: float,
    config: SarimaConfig,
    clip_lower: Optional[float] = None,
    clip_upper: Optional[float] = None,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    try:
        error_blocks = collect_backtest_error_blocks(
            y=y,
            steps=steps,
            initial_train_days=initial_train_days,
            step_days=step_days,
            config=config,
        )
    except ValueError:
        error_blocks = np.empty((0, steps))

    final_model = fit_sarima(y, config)
    forecast_result = final_model.get_forecast(steps=steps)
    forecast_mean = forecast_result.predicted_mean
    forecast_mean_values = forecast_mean.values.copy()

    if clip_lower is not None:
        forecast_mean_values = np.maximum(forecast_mean_values, clip_lower)
    if clip_upper is not None:
        forecast_mean_values = np.minimum(forecast_mean_values, clip_upper)

    if len(error_blocks) < MIN_BACKTEST_ERROR_BLOCKS:
        forecast_quantile_values = forecast_mean_values * (1 - SARIMA_NO_ERROR_HISTORY_ADJUSTMENT)
        if clip_lower is not None:
            forecast_quantile_values = np.maximum(forecast_quantile_values, clip_lower)
        if clip_upper is not None:
            forecast_quantile_values = np.minimum(forecast_quantile_values, clip_upper)

        output = pd.DataFrame(
            {
                "forecast_mean": forecast_mean_values,
                "forecast_quantile": forecast_quantile_values,
            },
            index=forecast_mean.index,
        )
        output.index.name = "forecast_date"
        return output.reset_index(), np.empty((0, steps)), error_blocks

    sampled_block_indexes = rng.integers(
        low=0,
        high=len(error_blocks),
        size=bootstrap_iterations,
    )
    sampled_errors = error_blocks[sampled_block_indexes]
    scenarios = forecast_mean_values.reshape(1, -1) + sampled_errors

    if clip_lower is not None:
        scenarios = np.maximum(scenarios, clip_lower)
    if clip_upper is not None:
        scenarios = np.minimum(scenarios, clip_upper)

    output = pd.DataFrame(
        {
            "forecast_mean": forecast_mean_values,
            "forecast_quantile": np.quantile(scenarios, quantile, axis=0),
        },
        index=forecast_mean.index,
    )
    output.index.name = "forecast_date"
    return output.reset_index(), scenarios, error_blocks

## 5. Прогноз для одной кассы

In [ ]:
def forecast_single_cashdesk(
    cashdesk_name: str,
    cashdesk_df: pd.DataFrame,
    config: SarimaConfig,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    saldo_series = make_regular_daily_series(
        cashdesk_df=cashdesk_df,
        value_col="saldo_turn",
        fill_method="zero",
    )
    need_series = make_regular_daily_series(
        cashdesk_df=cashdesk_df,
        value_col="cash_need",
        fill_method="zero",
    )

    if len(saldo_series) >= MIN_OBSERVATIONS:
        try:
            saldo_forecast = sarima_point_forecast(
                y=saldo_series,
                steps=FORECAST_DAYS,
                config=config,
            )
        except Exception:
            saldo_forecast = weekday_point_forecast(saldo_series, FORECAST_DAYS)
    else:
        saldo_forecast = weekday_point_forecast(saldo_series, FORECAST_DAYS)
    saldo_forecast.insert(0, "cashdesk_name", cashdesk_name)
    saldo_forecast.insert(1, "target", "saldo_turn")

    if len(need_series) >= MIN_OBSERVATIONS:
        try:
            need_forecast, need_scenarios, need_error_blocks = forecast_with_error_bootstrap(
                y=need_series,
                steps=FORECAST_DAYS,
                initial_train_days=INITIAL_TRAIN_DAYS,
                step_days=BACKTEST_STEP_DAYS,
                bootstrap_iterations=BOOTSTRAP_ITERATIONS,
                quantile=BOOTSTRAP_QUANTILE,
                config=config,
                clip_lower=CASH_NEED_CLIP_LOWER,
                clip_upper=CASH_NEED_CLIP_UPPER,
            )
        except Exception:
            need_forecast, need_scenarios, need_error_blocks = weekday_quantile_forecast(
                need_series,
                FORECAST_DAYS,
                BOOTSTRAP_QUANTILE,
            )
    else:
        need_forecast, need_scenarios, need_error_blocks = weekday_quantile_forecast(
            need_series,
            FORECAST_DAYS,
            BOOTSTRAP_QUANTILE,
        )
    need_forecast.insert(0, "cashdesk_name", cashdesk_name)
    need_forecast.insert(1, "target", "cash_need_negative_min_ns")

    return saldo_forecast, need_forecast, need_scenarios, need_error_blocks

## 6. Проверка на одной кассе

In [ ]:
cashdesk_stats = (
    daily_df
    .groupby("atdtmco_cashdesk_name")
    .agg(
        date_min=("calday", "min"),
        date_max=("calday", "max"),
        days_with_rows=("calday", "nunique"),
        saldo_turn_mean=("saldo_turn", "mean"),
        cash_need_mean=("cash_need", "mean"),
        cash_need_min=("cash_need", "min"),
        cash_need_max=("cash_need", "max"),
    )
    .sort_values("days_with_rows", ascending=False)
)

cashdesk_stats.head(20)

In [ ]:
test_cashdesk_name = cashdesk_stats.index[0]
print(f"Тестовая касса: {test_cashdesk_name}")

test_cashdesk_df = daily_df[daily_df["atdtmco_cashdesk_name"] == test_cashdesk_name].copy()

saldo_test_forecast, need_test_forecast, need_test_scenarios, need_test_error_blocks = forecast_single_cashdesk(
    cashdesk_name=test_cashdesk_name,
    cashdesk_df=test_cashdesk_df,
    config=SARIMA_CONFIG,
)

print("Прогноз saldo_turn:")
display(saldo_test_forecast)

print("Прогноз cash_need с выбранным bootstrap-квантилем:")
display(need_test_forecast)

print("Размер матрицы bootstrap-сценариев:", need_test_scenarios.shape)
print("Размер матрицы исторических блоков ошибок:", need_test_error_blocks.shape)

## 7. Запуск по всем кассам

In [ ]:
MAX_CASHDESKS_FOR_TEST = None

saldo_forecasts = []
need_forecasts = []
failed_cashdesks = []

cashdesk_groups = list(daily_df.groupby("atdtmco_cashdesk_name", sort=True))
if MAX_CASHDESKS_FOR_TEST is not None:
    cashdesk_groups = cashdesk_groups[:MAX_CASHDESKS_FOR_TEST]

for idx, (cashdesk_name, cashdesk_df) in enumerate(cashdesk_groups, start=1):
    print(f"[{idx}/{len(cashdesk_groups)}] Считаю кассу: {cashdesk_name}")

    try:
        saldo_forecast, need_forecast, _, _ = forecast_single_cashdesk(
            cashdesk_name=cashdesk_name,
            cashdesk_df=cashdesk_df.copy(),
            config=SARIMA_CONFIG,
        )
        saldo_forecasts.append(saldo_forecast)
        need_forecasts.append(need_forecast)
    except Exception as exc:
        failed_cashdesks.append(
            {
                "cashdesk_name": cashdesk_name,
                "error": str(exc),
                "rows": len(cashdesk_df),
                "date_min": cashdesk_df["calday"].min(),
                "date_max": cashdesk_df["calday"].max(),
            }
        )
        print(f"  Пропускаю кассу из-за ошибки: {exc}")

saldo_forecasts_df = pd.concat(saldo_forecasts, ignore_index=True) if saldo_forecasts else pd.DataFrame()
need_forecasts_df = pd.concat(need_forecasts, ignore_index=True) if need_forecasts else pd.DataFrame()
failed_cashdesks_df = pd.DataFrame(failed_cashdesks)

print("Готово")
print("saldo_forecasts_df:", saldo_forecasts_df.shape)
print("need_forecasts_df:", need_forecasts_df.shape)
print("failed_cashdesks_df:", failed_cashdesks_df.shape)

## 8. Формирование таблицы для выгрузки

In [ ]:
saldo_table_df = saldo_forecasts_df[
    ["cashdesk_name", "forecast_date", "forecast_mean"]
].rename(
    columns={
        "cashdesk_name": "atdtmco_cashdesk_name",
        "forecast_mean": "atdtmco_saldo_turn",
    }
)

ns_table_df = need_forecasts_df[
    ["cashdesk_name", "forecast_date", "forecast_quantile"]
].rename(
    columns={
        "cashdesk_name": "atdtmco_cashdesk_name",
        "forecast_quantile": "atdtmco_ns",
    }
)

forecast_table_df = saldo_table_df.merge(
    ns_table_df,
    on=["atdtmco_cashdesk_name", "forecast_date"],
    how="outer",
)
forecast_table_df.insert(0, "score_date", RUN_DATE)
forecast_table_df.insert(1, "report_date", MAX_DATA_DATE)
forecast_table_df = forecast_table_df[
    [
        "score_date",
        "report_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
        "atdtmco_saldo_turn",
        "atdtmco_ns",
    ]
]

forecast_table_df.head()

## 9. Сохранение результатов

In [ ]:
output_dir = Path("data/forecast_results")
output_dir.mkdir(parents=True, exist_ok=True)

forecast_table_df.to_csv(output_dir / "forecast_table.csv", index=False)
saldo_forecasts_df.to_csv(output_dir / "saldo_forecasts.csv", index=False)
need_forecasts_df.to_csv(output_dir / "need_forecasts_bootstrap.csv", index=False)
inactive_cashdesks_df.to_csv(output_dir / "inactive_cashdesks.csv", index=False)
failed_cashdesks_df.to_csv(output_dir / "failed_cashdesks.csv", index=False)

print(f"Результаты сохранены в: {output_dir.resolve()}")

## 11. Ретро-прогон для оценки качества

Блок имитирует ежедневный запуск на историческую дату и сравнивает прогноз с фактом за следующий месяц.

In [ ]:
RETRO_RUN_DATE = pd.Timestamp("2025-05-01")
RETRO_EVALUATION_DAYS = 30
RETRO_MAX_DATA_DATE = RETRO_RUN_DATE - pd.Timedelta(days=DATA_LAG_DAYS)
RETRO_DATE_FROM = (RETRO_MAX_DATA_DATE - pd.DateOffset(months=HISTORY_MONTHS)).date().isoformat()
RETRO_TRAIN_DATE_TO_EXCLUSIVE = (RETRO_MAX_DATA_DATE + pd.Timedelta(days=1)).date().isoformat()
RETRO_FACT_DATE_TO_EXCLUSIVE = (RETRO_RUN_DATE + pd.Timedelta(days=RETRO_EVALUATION_DAYS)).date().isoformat()
RETRO_MODEL_STEPS = RETRO_EVALUATION_DAYS + DATA_LAG_DAYS - 1

retro_query = f"""
select
    atdtmco_cashdesk_name,
    atdtmco_calday,
    atdtmco_saldo_turn,
    atdtmco_ns
from {SOURCE_TABLE}
where atdtmco_calday >= date '{RETRO_DATE_FROM}'
  and atdtmco_calday < date '{RETRO_FACT_DATE_TO_EXCLUSIVE}'
"""

for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
    retro_query += (
        f"\n  and not ("
        f"atdtmco_calday >= date '{exclude_start}' "
        f"and atdtmco_calday < date '{exclude_end}'"
        f")"
    )

if CASHDESK_FILTER:
    escaped_names = ", ".join(["'" + name.replace("'", "''") + "'" for name in CASHDESK_FILTER])
    retro_query += f"\n  and atdtmco_cashdesk_name in ({escaped_names})"

with engine_cdw.connect() as conn:
    retro_raw_df = pd.read_sql(text(retro_query), conn)

retro_raw_df.columns = retro_raw_df.columns.str.lower()
retro_raw_df["atdtmco_calday"] = pd.to_datetime(retro_raw_df["atdtmco_calday"])

retro_daily_df = (
    retro_raw_df
    .assign(calday=lambda df: df["atdtmco_calday"].dt.normalize())
    .groupby(["atdtmco_cashdesk_name", "calday"], as_index=False)
    .agg(
        saldo_turn=("atdtmco_saldo_turn", "sum"),
        ns_daily_min=("atdtmco_ns", "min"),
    )
)
retro_daily_df["cash_need"] = retro_daily_df["ns_daily_min"].clip(upper=0)
retro_daily_df = retro_daily_df.sort_values(["atdtmco_cashdesk_name", "calday"]).reset_index(drop=True)

retro_active_date_from = RETRO_MAX_DATA_DATE - pd.DateOffset(months=ACTIVE_LOOKBACK_MONTHS)
retro_active_cashdesks = retro_daily_df.loc[
    (retro_daily_df["calday"] >= retro_active_date_from)
    & (retro_daily_df["calday"] <= RETRO_MAX_DATA_DATE),
    "atdtmco_cashdesk_name",
].drop_duplicates()

retro_inactive_cashdesks_df = (
    retro_daily_df.loc[~retro_daily_df["atdtmco_cashdesk_name"].isin(retro_active_cashdesks), ["atdtmco_cashdesk_name"]]
    .drop_duplicates()
    .assign(score_date=RETRO_RUN_DATE, report_date=RETRO_MAX_DATA_DATE)
)
retro_daily_df = retro_daily_df[retro_daily_df["atdtmco_cashdesk_name"].isin(retro_active_cashdesks)].reset_index(drop=True)

retro_train_df = retro_daily_df[retro_daily_df["calday"] <= RETRO_MAX_DATA_DATE].copy()
retro_actual_df = (
    retro_daily_df[
        (retro_daily_df["calday"] >= RETRO_RUN_DATE)
        & (retro_daily_df["calday"] < pd.Timestamp(RETRO_FACT_DATE_TO_EXCLUSIVE))
    ]
    [["atdtmco_cashdesk_name", "calday", "saldo_turn", "cash_need"]]
    .rename(
        columns={
            "atdtmco_cashdesk_name": "cashdesk_name",
            "calday": "forecast_date",
            "saldo_turn": "actual_saldo_turn",
            "cash_need": "actual_cash_need",
        }
    )
)


def forecast_single_cashdesk_retro(
    cashdesk_name: str,
    cashdesk_df: pd.DataFrame,
    steps: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    saldo_series = make_regular_daily_series(
        cashdesk_df=cashdesk_df,
        value_col="saldo_turn",
        fill_method="zero",
    )
    need_series = make_regular_daily_series(
        cashdesk_df=cashdesk_df,
        value_col="cash_need",
        fill_method="zero",
    )

    if len(saldo_series) >= MIN_OBSERVATIONS:
        try:
            saldo_forecast = sarima_point_forecast(
                y=saldo_series,
                steps=steps,
                config=SARIMA_CONFIG,
            )
        except Exception:
            saldo_forecast = weekday_point_forecast(saldo_series, steps)
    else:
        saldo_forecast = weekday_point_forecast(saldo_series, steps)
    saldo_forecast.insert(0, "cashdesk_name", cashdesk_name)

    if len(need_series) >= MIN_OBSERVATIONS:
        try:
            need_forecast, _, _ = forecast_with_error_bootstrap(
                y=need_series,
                steps=steps,
                initial_train_days=INITIAL_TRAIN_DAYS,
                step_days=BACKTEST_STEP_DAYS,
                bootstrap_iterations=BOOTSTRAP_ITERATIONS,
                quantile=BOOTSTRAP_QUANTILE,
                config=SARIMA_CONFIG,
                clip_lower=CASH_NEED_CLIP_LOWER,
                clip_upper=CASH_NEED_CLIP_UPPER,
            )
        except Exception:
            need_forecast, _, _ = weekday_quantile_forecast(need_series, steps, BOOTSTRAP_QUANTILE)
    else:
        need_forecast, _, _ = weekday_quantile_forecast(need_series, steps, BOOTSTRAP_QUANTILE)
    need_forecast.insert(0, "cashdesk_name", cashdesk_name)

    return saldo_forecast, need_forecast


retro_saldo_forecasts = []
retro_need_forecasts = []
retro_failed_cashdesks = []

for idx, (cashdesk_name, cashdesk_df) in enumerate(retro_train_df.groupby("atdtmco_cashdesk_name", sort=True), start=1):
    print(f"[{idx}/{retro_train_df['atdtmco_cashdesk_name'].nunique()}] Ретро-расчет кассы: {cashdesk_name}")

    try:
        saldo_forecast, need_forecast = forecast_single_cashdesk_retro(
            cashdesk_name=cashdesk_name,
            cashdesk_df=cashdesk_df.copy(),
            steps=RETRO_MODEL_STEPS,
        )
        retro_saldo_forecasts.append(saldo_forecast)
        retro_need_forecasts.append(need_forecast)
    except Exception as exc:
        retro_failed_cashdesks.append(
            {
                "cashdesk_name": cashdesk_name,
                "error": str(exc),
                "rows": len(cashdesk_df),
                "date_min": cashdesk_df["calday"].min(),
                "date_max": cashdesk_df["calday"].max(),
            }
        )
        print(f"  Пропускаю кассу из-за ошибки: {exc}")

retro_saldo_forecasts_df = pd.concat(retro_saldo_forecasts, ignore_index=True) if retro_saldo_forecasts else pd.DataFrame()
retro_need_forecasts_df = pd.concat(retro_need_forecasts, ignore_index=True) if retro_need_forecasts else pd.DataFrame()
retro_failed_cashdesks_df = pd.DataFrame(retro_failed_cashdesks)

retro_eval_start = RETRO_RUN_DATE
retro_eval_end = pd.Timestamp(RETRO_FACT_DATE_TO_EXCLUSIVE)

retro_saldo_forecasts_df = retro_saldo_forecasts_df[
    (retro_saldo_forecasts_df["forecast_date"] >= retro_eval_start)
    & (retro_saldo_forecasts_df["forecast_date"] < retro_eval_end)
].rename(
    columns={
        "cashdesk_name": "atdtmco_cashdesk_name",
        "forecast_mean": "atdtmco_saldo_turn",
    }
)

retro_need_forecasts_df = retro_need_forecasts_df[
    (retro_need_forecasts_df["forecast_date"] >= retro_eval_start)
    & (retro_need_forecasts_df["forecast_date"] < retro_eval_end)
].rename(
    columns={
        "cashdesk_name": "atdtmco_cashdesk_name",
        "forecast_quantile": "atdtmco_ns",
    }
)

retro_forecast_table_df = (
    retro_saldo_forecasts_df
    .merge(
        retro_need_forecasts_df[["atdtmco_cashdesk_name", "forecast_date", "atdtmco_ns"]],
        on=["atdtmco_cashdesk_name", "forecast_date"],
        how="outer",
    )
)
retro_forecast_table_df.insert(0, "score_date", RETRO_RUN_DATE)
retro_forecast_table_df.insert(1, "report_date", RETRO_MAX_DATA_DATE)
retro_forecast_table_df = retro_forecast_table_df[
    [
        "score_date",
        "report_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
        "atdtmco_saldo_turn",
        "atdtmco_ns",
    ]
]

retro_actual_table_df = retro_actual_df.rename(
    columns={
        "cashdesk_name": "atdtmco_cashdesk_name",
        "actual_saldo_turn": "fact_atdtmco_saldo_turn",
        "actual_cash_need": "fact_atdtmco_ns",
    }
)

retro_result_df = retro_forecast_table_df.merge(
    retro_actual_table_df,
    on=["atdtmco_cashdesk_name", "forecast_date"],
    how="left",
)
retro_result_df["error_atdtmco_saldo_turn"] = retro_result_df["fact_atdtmco_saldo_turn"] - retro_result_df["atdtmco_saldo_turn"]
retro_result_df["error_atdtmco_ns"] = retro_result_df["fact_atdtmco_ns"] - retro_result_df["atdtmco_ns"]
retro_result_df = retro_result_df[
    [
        "score_date",
        "report_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
        "atdtmco_saldo_turn",
        "atdtmco_ns",
        "fact_atdtmco_saldo_turn",
        "fact_atdtmco_ns",
        "error_atdtmco_saldo_turn",
        "error_atdtmco_ns",
    ]
]

retro_output_dir = Path("data/forecast_results/retro")
retro_output_dir.mkdir(parents=True, exist_ok=True)
retro_result_df.to_csv(retro_output_dir / f"retro_result_{RETRO_RUN_DATE.date()}.csv", index=False)
retro_inactive_cashdesks_df.to_csv(retro_output_dir / f"retro_inactive_cashdesks_{RETRO_RUN_DATE.date()}.csv", index=False)
retro_failed_cashdesks_df.to_csv(retro_output_dir / f"retro_failed_cashdesks_{RETRO_RUN_DATE.date()}.csv", index=False)

retro_metrics = pd.DataFrame(
    [
        {
            "score_date": RETRO_RUN_DATE,
            "report_date": RETRO_MAX_DATA_DATE,
            "rows": len(retro_result_df),
            "saldo_turn_mae": retro_result_df["error_atdtmco_saldo_turn"].abs().mean(),
            "ns_mae": retro_result_df["error_atdtmco_ns"].abs().mean(),
            "failed_cashdesks": len(retro_failed_cashdesks_df),
        }
    ]
)

retro_metrics

## 10. Описание результатов

См. `README_cashdesk_sarima_bootstrap.md`.